In [1]:
# Install required ffprobe package for video metadata extraction
!pip install ffprobe-python

In [2]:
import cv2
import pandas as pd
import os
import json
import io
import shutil
import numpy as np
import math
import zipfile
import subprocess
import traceback
from datetime import datetime, timedelta, timezone
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Define Standard Paths
BASE_DRIVE_DIR = '/content/drive/MyDrive/Smartphone_SensorData'
PROCESSED_DIR = '/content/drive/MyDrive/Processed'
TRACKER_CSV = os.path.join(BASE_DRIVE_DIR, 'scanning.csv')

# Ensure the required directories exist
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(BASE_DRIVE_DIR, exist_ok=True)

Mounted at /content/drive


In [3]:
def get_video_start_time(video_path):
    """
    Reads video metadata using ffprobe to find the absolute start time,
    correctly handling timezone information.
    """
    print("Reading video metadata to find absolute start time...")
    try:
        cmd = ["ffprobe", "-v", "quiet", "-print_format", "json", "-show_format", video_path]
        result = subprocess.run(cmd, capture_output=True, text=True, check=True)
        metadata = json.loads(result.stdout)

        # This is the most likely tag for timezone-aware time
        creation_time_str = metadata['format']['tags']['creation_time']

        # datetime.fromisoformat() is powerful and handles 'Z' or '+XX:XX' offsets
        video_start_time = datetime.fromisoformat(creation_time_str)
        print(f"Successfully parsed video start time: {video_start_time.isoformat()}")
        return video_start_time

    except (subprocess.CalledProcessError, KeyError, FileNotFoundError, ValueError) as e:
        print(f"Warning: Could not extract creation_time from video metadata: {e}")
        print("Falling back to file modification time (less accurate).")
        try:
            mtime = os.path.getmtime(video_path)
            # Create a timezone-aware datetime object from the local file time
            video_start_time = datetime.fromtimestamp(mtime).astimezone()
            print(f"Using fallback file modification time: {video_start_time.isoformat()}")
            return video_start_time
        except Exception as file_e:
            print(f"Could not even get file modification time: {file_e}. Aborting.")
            return None


def extract_video_timestamps_and_frames(video_path, output_dir):
    """
    Extracts each frame from a video and saves it with a precise UTC timestamp.
    This function reads the video's creation time from metadata and uses
    frame-specific millisecond offsets to calculate the real-world time for each frame.
    """
    print("Reading video metadata to find absolute start time...")
    try:
        cmd = ["ffprobe", "-v", "quiet", "-print_format", "json", "-show_format", video_path]
        result = subprocess.run(cmd, capture_output=True, text=True, check=True)
        metadata = json.loads(result.stdout)

        creation_time_str = metadata['format']['tags']['creation_time']

        if creation_time_str.endswith('Z'):
            video_start_utc = datetime.fromisoformat(creation_time_str[:-1]).replace(tzinfo=timezone.utc)
        else:
            print("Warning: Assuming UTC for start time as 'Z' was not found.")
            video_start_utc = datetime.fromisoformat(creation_time_str).replace(tzinfo=timezone.utc)

        print(f"Video start time (UTC): {video_start_utc.isoformat()}")

    except (subprocess.CalledProcessError, KeyError, FileNotFoundError) as e:
        print(f"Error: Could not extract creation_time from video metadata: {e}")
        print("As a fallback, using the file's modification time (less accurate).")
        try:
            mtime = os.path.getmtime(video_path)
            video_start_utc = datetime.fromtimestamp(mtime, tz=timezone.utc)
            print(f"Using fallback file modification time (UTC): {video_start_utc.isoformat()}")
        except Exception as file_e:
            print(f"Could not even get file modification time: {file_e}. Aborting.")
            return

    print("\nExtracting frames and calculating precise timestamps...")
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Error: Could not open video file '{video_path}'")
        return

    os.makedirs(output_dir, exist_ok=True)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    frame_index = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        msec_offset = cap.get(cv2.CAP_PROP_POS_MSEC)
        current_frame_utc = video_start_utc + timedelta(milliseconds=msec_offset)
        timestamp_str_folder = current_frame_utc.isoformat().replace(':', '-')

        frame_output_dir = os.path.join(output_dir, timestamp_str_folder)
        os.makedirs(frame_output_dir, exist_ok=True)

        image_path = os.path.join(frame_output_dir, 'frame.jpg')
        cv2.imwrite(image_path, frame)

        frame_metadata = {
            "utc_timestamp": current_frame_utc.isoformat(),
            "video_start_utc": video_start_utc.isoformat(),
            "frame_offset_msec": msec_offset
        }
        json_path = os.path.join(frame_output_dir, 'frame_metadata.json')
        with open(json_path, 'w') as f:
            json.dump(frame_metadata, f, indent=4)

        if frame_index % 100 == 0:
            print(f"  Processed frame {frame_index}/{total_frames} -> {timestamp_str_folder}")

        frame_index += 1

    cap.release()
    print("\nFinished extracting all frames with precise timestamps.")


def link_sensors_to_frames(video_path, frames_base_dir, imu_data, gyro_data, gps_data, time_window_sec=5.0):
    """
    Correctly links sensor data by calculating the offset between video and sensor
    clocks, correctly handling local timezones by converting all to UTC.
    """
    print("\nStarting TIMEZONE-AWARE sensor synchronization...")

    video_start_time = get_video_start_time(video_path)
    if video_start_time is None:
        return

    first_sensor_time = None
    if not gps_data.empty and 'local_time' in gps_data.columns:
        try:
            sorted_gps = gps_data.sort_values(by='time').reset_index()
            first_local_time_str = sorted_gps.loc[0, 'local_time']
            first_sensor_time = datetime.fromisoformat(first_local_time_str)
        except (KeyError, ValueError, IndexError) as e:
            print(f"ERROR: Could not parse 'local_time' from GPS data: {e}")
            return
    else:
        print("ERROR: GPS data is empty or missing 'local_time' column. Cannot sync.")
        return

    time_offset = video_start_time - first_sensor_time

    print(f"\n--- Clock Synchronization (Timezone-Aware) ---")
    print(f"Video Start Time: {video_start_time.isoformat()}")
    print(f"First Sensor Event: {first_sensor_time.isoformat()}")
    print(f"Calculated Offset (Video - Sensor): {time_offset}")
    print("Applying this offset to all sensor timestamps...")

    all_dfs = {'imu': imu_data, 'gyro': gyro_data, 'gps': gps_data}
    for df in all_dfs.values():
        if 'time' in df.columns:
            base_time = pd.to_datetime(df['time'], unit='ns')
            df['utc_datetime_aligned'] = (base_time + time_offset).dt.tz_localize('UTC')

    subdirs = sorted([d for d in os.listdir(frames_base_dir) if os.path.isdir(os.path.join(frames_base_dir, d))])

    for i, dirname in enumerate(subdirs):
        folder_path = os.path.join(frames_base_dir, dirname)
        metadata_path = os.path.join(folder_path, 'frame_metadata.json')
        if not os.path.exists(metadata_path): continue

        with open(metadata_path, 'r') as f: frame_meta = json.load(f)

        frame_time = datetime.fromisoformat(frame_meta['utc_timestamp'])

        start_time = frame_time
        end_time = frame_time + timedelta(seconds=time_window_sec)

        imu_window_df = imu_data[(imu_data['utc_datetime_aligned'] >= start_time) & (imu_data['utc_datetime_aligned'] < end_time)]
        gyro_window_df = gyro_data[(gyro_data['utc_datetime_aligned'] >= start_time) & (gyro_data['utc_datetime_aligned'] < end_time)]
        gps_window_df = gps_data[(gps_data['utc_datetime_aligned'] >= start_time) & (gps_data['utc_datetime_aligned'] < end_time)]

        output_data = {
            "frame_timestamp_utc": frame_time.isoformat(),
            "time_window_sec": time_window_sec,
            "imu": imu_window_df.drop(columns=['utc_datetime_aligned'], errors='ignore').to_dict(orient='records'),
            "gyro": gyro_window_df.drop(columns=['utc_datetime_aligned'], errors='ignore').to_dict(orient='records'),
            "gps": gps_window_df.drop(columns=['utc_datetime_aligned'], errors='ignore').to_dict(orient='records')
        }

        json_path = os.path.join(folder_path, 'sensor_data.json')
        with open(json_path, 'w') as f:
            json.dump(output_data, f, indent=4)

    print("\nFinished linking sensor data to all frames.")

In [4]:
# 1. Load or Initialize Tracker CSV
if os.path.exists(TRACKER_CSV):
    df_tracker = pd.read_csv(TRACKER_CSV)
    processed_files = set(df_tracker['processed_file'].tolist())
    print(f"Tracker loaded. Found {len(processed_files)} previously processed files.")
else:
    processed_files = set()
    df_tracker = pd.DataFrame(columns=['processed_file'])
    df_tracker.to_csv(TRACKER_CSV, index=False)
    print("Tracker not found. Created a new scanning.csv.")

# 2. Scan Target Folder for New Files
all_files = os.listdir(BASE_DRIVE_DIR)
zip_files = [f for f in all_files if f.endswith('.zip')]
new_zip_files = [f for f in zip_files if f not in processed_files]

print(f"Found {len(new_zip_files)} new files to process.")

# 3. Processing Loop
for zip_filename in new_zip_files:
    print(f"\n=======================================================")
    print(f"--- Processing {zip_filename} ---")
    zip_path = os.path.join(BASE_DRIVE_DIR, zip_filename)

    extract_dir = '/content/temp_extract'
    frames_output_dir = '/content/temp_frames_output'

    # Clean temp folders from previous iterations
    for temp_dir in [extract_dir, frames_output_dir]:
        if os.path.exists(temp_dir): shutil.rmtree(temp_dir)
        os.makedirs(temp_dir)

    try:
        # Unzip locally
        print(f"Unzipping {zip_filename}...")
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_dir)

        # Dynamically locate the video and required CSV files inside the extracted folder
        video_file, imu_file, gyro_file, gps_file = None, None, None, None

        # Walk to handle subdirectories inside the zip
        for root, dirs, files in os.walk(extract_dir):
            for file in files:
                lower_name = file.lower()
                if lower_name.endswith('.mp4'): video_file = os.path.join(root, file)
                elif 'accelerometer.csv' in lower_name: imu_file = os.path.join(root, file)
                elif 'gyroscope.csv' in lower_name: gyro_file = os.path.join(root, file)
                elif 'location.csv' in lower_name: gps_file = os.path.join(root, file)

        if not all([video_file, imu_file, gyro_file, gps_file]):
            print(f"Missing essential files in {zip_filename}. Skipping...")
            continue

        # Load Sensor Data DataFrames
        print("Loading sensor data into Pandas...")
        imu_df = pd.read_csv(imu_file, skipinitialspace=True)
        gyro_df = pd.read_csv(gyro_file, skipinitialspace=True)
        gps_df = pd.read_csv(gps_file, skipinitialspace=True)

        # Step 1: Extract Frames
        extract_video_timestamps_and_frames(video_file, frames_output_dir)

        # Step 2: Sync and Extract Next 5-seconds of Sensor Data per frame
        link_sensors_to_frames(video_file, frames_output_dir, imu_df, gyro_df, gps_df, time_window_sec=5.0)

        # Step 3: Archive Processed Data & Move to Drive
        print("Archiving processed outputs...")
        processed_zip_name = f"processed_{zip_filename}"
        processed_zip_path_temp = f"/content/{processed_zip_name}"

        # Create ZIP locally (faster) then copy to Google Drive
        shutil.make_archive(processed_zip_path_temp.replace('.zip', ''), 'zip', frames_output_dir)
        shutil.move(processed_zip_path_temp, os.path.join(PROCESSED_DIR, processed_zip_name))

        print(f"Saved processed data to: {os.path.join(PROCESSED_DIR, processed_zip_name)}")

        # Update Tracker
        new_row = pd.DataFrame({'processed_file': [zip_filename]})
        df_tracker = pd.concat([df_tracker, new_row], ignore_index=True)
        df_tracker.to_csv(TRACKER_CSV, index=False)
        print(f"SUCCESS: Successfully processed and tracked {zip_filename}")

    except Exception as e:
        print(f"ERROR: Failed processing {zip_filename}: {e}")
        traceback.print_exc()

    finally:
        # Cleanup
        for temp_dir in [extract_dir, frames_output_dir]:
            if os.path.exists(temp_dir): shutil.rmtree(temp_dir)

print("\nBatch processing complete.")

Tracker not found. Created a new scanning.csv.
Found 53 new files to process.

--- Processing A9-2026-01-10_15-07-26-android.zip ---
Unzipping A9-2026-01-10_15-07-26-android.zip...
Loading sensor data into Pandas...
Reading video metadata to find absolute start time...
Video start time (UTC): 2026-01-10T15:08:56+00:00

Extracting frames and calculating precise timestamps...
  Processed frame 0/2641 -> 2026-01-10T15-08-56+00-00
  Processed frame 100/2641 -> 2026-01-10T15-09-00.480856+00-00
  Processed frame 200/2641 -> 2026-01-10T15-09-03.813322+00-00
  Processed frame 300/2641 -> 2026-01-10T15-09-07.145811+00-00
  Processed frame 400/2641 -> 2026-01-10T15-09-10.478322+00-00
  Processed frame 500/2641 -> 2026-01-10T15-09-13.800778+00-00
  Processed frame 600/2641 -> 2026-01-10T15-09-17.143289+00-00
  Processed frame 700/2641 -> 2026-01-10T15-09-20.509089+00-00
  Processed frame 800/2641 -> 2026-01-10T15-09-23.831578+00-00
  Processed frame 900/2641 -> 2026-01-10T15-09-27.164056+00-00
  